In [ ]:
%load_ext autoreload
%autoreload 2

import polars as pl
from anngeno import AnnGeno

In [ ]:
pg = pl.read_parquet('/home/dnanexus/data_dir/250717_proteingym_SNP_DMS_scores_human_coding_genes.parquet')
pg

In [ ]:
anno = pl.read_parquet('/home/dnanexus/data_dir/genebass_1e6_coding_variants.ag/annotations.parquet').with_columns(
    pl.when(
        pl.col('amino_acids').is_not_null() & pl.col('protein_position').is_not_null()
    ).then(
        pl.col('amino_acids').str.split('/').list.get(0) +
        pl.col('protein_position').str.split('/').list.get(0) +
        pl.col('amino_acids').str.split('/').list.get(1)
    ).otherwise(None).alias('mutant')
)

anno

In [ ]:
pg.filter(pl.col('gene_name').is_in(['GCK', 'BRCA1']))['file_name'].value_counts().sort('count', descending=True)

In [ ]:
pg_rap = pg.filter(pl.col('file_name').is_in(['BRCA1_HUMAN_Findlay_2018', 'HXK4_HUMAN_Gersing_2022_activity']))[['mutant', 'dms_score', 'gene_name', 'file_name', 'region']].join(anno, on=['region', 'mutant'], how='right')
pg_rap